In [0]:
from __future__ import annotations
import requests
from datetime import datetime

In [0]:
TENANT_ID     = dbutils.secrets.get(scope="compliance-grc", key="sharepoint-tenant-id")
CLIENT_ID     = dbutils.secrets.get(scope="compliance-grc", key="sharepoint-client-id")
CLIENT_SECRET = dbutils.secrets.get(scope="compliance-grc", key="sharepoint-client-secret")

SITE_HOSTNAME = "cercbr.sharepoint.com"
SITE_PATH     = "sites/BETAFerramentaGRC"
LIST_NAME     = "CONTINUOUS_AUDIT_TRIGGER_DETAILS"

In [0]:
def update_sharepoint_trigger(
    *,
    test_name: str,
    description: str,
    responsible_area: str,
    risk_id: str,
    threshold: int,
    incident_count: int,
    execution_date: str | None = None,
) -> None:
    """
    Cria um item na lista do SharePoint com o disparo do teste.
    Deve ser chamada SOMENTE quando o teste falhar em PRD.
    """
    try:
        token = requests.post(
            f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",
            data={
                "client_id": CLIENT_ID,
                "client_secret": CLIENT_SECRET,
                "grant_type": "client_credentials",
                "scope": "https://graph.microsoft.com/.default",
            },
            timeout=20,
        ).json()["access_token"]

        headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

        site_id = requests.get(
            f"https://graph.microsoft.com/v1.0/sites/{SITE_HOSTNAME}:/{SITE_PATH}:",
            headers=headers, timeout=20
        ).json()["id"]

        lists = requests.get(
            f"https://graph.microsoft.com/v1.0/sites/{site_id}/lists",
            headers=headers, timeout=20
        ).json()["value"]

        list_id = next(l["id"] for l in lists if l["displayName"] == LIST_NAME)

        fields = {
            "TEST_NAME":        test_name,
            "DESCRIPTION":      description,
            "RESPONSIBLE_AREA": responsible_area,
            "RISK_ID":          risk_id,
            "EXECUTION_DATE":   execution_date or datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "THRESHOLD":        threshold,
            "INCIDENT_COUNT":   incident_count,
        }

        resp = requests.post(
            f"https://graph.microsoft.com/v1.0/sites/{site_id}/lists/{list_id}/items",
            headers=headers, json={"fields": fields}, timeout=20
        )

        if resp.status_code not in (200, 201):
            print(f"⚠️ Atualização do SharePoint falhou [{resp.status_code}]: {resp.text}")
        else:
            print(f"✅ SharePoint atualizado: {test_name}")

    except Exception as e:
        print(f"⚠️ Falha ao atualizar SharePoint: {e}")